# 04 — Kafka Pipeline End-to-End Test

**Objective:** Test the entire online streaming detection pipeline without requiring a live Kafka broker — using mock/simulated message processing to validate:
- Message serialisation / deserialisation
- Rolling 60-step buffer per sensor
- Autoencoder inference latency (< 50ms target)
- Anomaly alert generation
- FastAPI `/predict` endpoint (with `httpx` test client)

**Note:** Sections 1-4 work fully offline. Section 5 tests the API and requires the model to be trained (notebook 03).

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

from src.data.generator import IoTDataGenerator
from src.streaming.kafka_config import SENSOR_TOPIC, ALERT_TOPIC, BOOTSTRAP_SERVERS

sns.set_style("darkgrid")
print("Imports OK")
print(f"Kafka topics:")
print(f"  Sensor  : {SENSOR_TOPIC}")
print(f"  Alerts  : {ALERT_TOPIC}")

## 1. Message Serialisation Test

In [ ]:
gen = IoTDataGenerator(n_sensors=50, seq_length=60)
seq = gen.generate_normal_sequence()  # (60, 50)

# Simulate a single timestep message (as the producer would send)
message = {
    "timestamp": time.time(),
    "sensor_id": 7,
    "values":    seq[0, :].tolist(),   # one timestep, all 50 sensors
}

# Serialise → bytes → deserialise
encoded = json.dumps(message).encode("utf-8")
decoded = json.loads(encoded.decode("utf-8"))

assert decoded["sensor_id"] == message["sensor_id"]
assert len(decoded["values"]) == 50
assert abs(decoded["timestamp"] - message["timestamp"]) < 0.001

print(f"✅ Message serialisation OK")
print(f"   sensor_id : {decoded['sensor_id']}")
print(f"   values    : [{decoded['values'][0]:.4f}, ..., {decoded['values'][-1]:.4f}]")
print(f"   size      : {len(encoded)} bytes")

## 2. Rolling Buffer Logic Test

Simulates the consumer's per-sensor buffer that accumulates 60 timesteps antes de inferir.

In [ ]:
SEQ_LENGTH = 60
N_SENSORS  = 50

class MockConsumer:
    def __init__(self, seq_length: int = SEQ_LENGTH):
        self.seq_length = seq_length
        self.buffers = defaultdict(list)
        self.inferences = 0
    
    def process(self, sensor_id: int, values: list) -> bool:
        self.buffers[sensor_id].append(values)
        if len(self.buffers[sensor_id]) >= self.seq_length:
            self.buffers[sensor_id] = self.buffers[sensor_id][-self.seq_length:]
            self.inferences += 1
            return True
        return False

consumer = MockConsumer()
inject_at = 100  # inject anomaly at step 100

inference_steps = []
for step in range(300):
    for sid in range(5):  # simulate 5 sensors for speed
        normal = gen.generate_normal_sequence()
        if step == inject_at and sid == 2:
            seq_ = gen.inject_anomaly(normal, anomaly_type="spike")
            values = seq_[step % 60, :].tolist()
        else:
            values = normal[step % 60, :].tolist()
        ready = consumer.process(sid, values)
        if ready:
            inference_steps.append(step)

print(f"✅ Buffer test complete")
print(f"   Total inferences triggered : {consumer.inferences}")
print(f"   Buffers maintained         : {len(consumer.buffers)} sensors")
for sid, buf in consumer.buffers.items():
    print(f"   Sensor {sid}: buffer len = {len(buf)}  (should be {SEQ_LENGTH})")

## 3. Inference Latency Benchmark

Measures autoencoder inference time per sequence to validate the **< 50ms target**.

In [ ]:
import joblib

MODEL_PATH     = "../models/bilstm_autoencoder.h5"
SCALER_PATH    = "../models/scaler.pkl"
THRESHOLD_PATH = "../models/threshold.pkl"

if not os.path.exists(MODEL_PATH):
    print("⚠️  Model not found. Run notebook 03_training.ipynb first.")
else:
    import tensorflow as tf
    keras_model = tf.keras.models.load_model(MODEL_PATH)
    scaler      = joblib.load(SCALER_PATH)
    threshold   = float(joblib.load(THRESHOLD_PATH))

    N_RUNS = 50
    latencies = []

    for _ in range(N_RUNS):
        seq = gen.generate_normal_sequence()  # (60, 50)
        scaled = scaler.transform(seq.reshape(-1, 50)).reshape(1, 60, 50).astype(np.float32)

        t0 = time.perf_counter()
        recon = keras_model.predict(scaled, verbose=0)
        latency_ms = (time.perf_counter() - t0) * 1000
        latencies.append(latency_ms)

    latencies = np.array(latencies)
    print(f"\nInference latency over {N_RUNS} runs:")
    print(f"  mean   : {latencies.mean():.1f} ms")
    print(f"  p50    : {np.percentile(latencies, 50):.1f} ms")
    print(f"  p95    : {np.percentile(latencies, 95):.1f} ms")
    print(f"  p99    : {np.percentile(latencies, 99):.1f} ms")
    print(f"  target : < 50 ms")

    under_50ms = (latencies < 50).mean() * 100
    print(f"\n  {under_50ms:.1f}% of inferences under 50ms")
    if latencies.mean() < 50:
        print("  ✅ Latency target achieved")
    else:
        print("  ⚠️  Latency exceeds 50ms mean — consider TF optimization or batch inference")

    # Histogram
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(latencies, bins=20, color="#4CAF50", alpha=0.85, edgecolor="white")
    ax.axvline(50, color="red", linestyle="--", linewidth=2, label="50ms target")
    ax.axvline(latencies.mean(), color="orange", linestyle="-", linewidth=2,
               label=f"Mean = {latencies.mean():.1f}ms")
    ax.set_xlabel("Inference time (ms)")
    ax.set_ylabel("Count")
    ax.set_title("Autoencoder Inference Latency Distribution", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../logs/kafka_latency_benchmark.png", dpi=120, bbox_inches="tight")
    plt.show()

## 4. End-to-End Anomaly Detection Simulation

Simulates 50 sensors over 120 timesteps, injecting anomalies, and tracks detections.

In [ ]:
if not os.path.exists(MODEL_PATH):
    print("⚠️  Model not found. Run notebook 03_training.ipynb first.")
else:
    N_STREAMS = 5         # simulate 5 sensors for speed in notebook
    N_STEPS   = 120       # timesteps to simulate
    INJECT_AT = [70, 85]  # inject anomalies at these steps

    buffers   = defaultdict(list)
    errors_by_step = {sid: [] for sid in range(N_STREAMS)}
    alerts    = []

    for step in range(N_STEPS):
        for sid in range(N_STREAMS):
            normal = gen.generate_normal_sequence()
            if step in INJECT_AT and sid == 2:
                values = gen.inject_anomaly(normal, anomaly_type="spike")[step % 60, :].tolist()
            else:
                values = normal[step % 60, :].tolist()

            buffers[sid].append(values)

            if len(buffers[sid]) >= SEQ_LENGTH:
                buffers[sid] = buffers[sid][-SEQ_LENGTH:]
                seq = np.array(buffers[sid], dtype=np.float32)  # (60, 50)
                scaled = scaler.transform(seq.reshape(-1, 50)).reshape(1, 60, 50).astype(np.float32)
                recon = keras_model.predict(scaled, verbose=0)
                error = float(np.mean((scaled - recon)**2))
                errors_by_step[sid].append((step, error))

                if error > threshold:
                    alerts.append({
                        "step": step, "sensor_id": sid,
                        "error": error, "threshold": threshold,
                        "severity": "HIGH" if error > 2*threshold else "MEDIUM"
                    })
                    print(f"  🚨 ALERT step={step:3d} sensor={sid}  "
                          f"error={error:.4f}  severity={alerts[-1]['severity']}")

    print(f"\n✅ Simulation complete:")
    print(f"   Total steps    : {N_STEPS}")
    print(f"   Alerts fired   : {len(alerts)}")
    print(f"   Injected at    : steps {INJECT_AT} (sensor 2)")

    # Plot error timeline for sensor 2
    steps2, errs2 = zip(*errors_by_step[2]) if errors_by_step[2] else ([], [])
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(steps2, errs2, color="#2196F3", lw=1.5, label="Sensor 2 reconstruction error")
    ax.axhline(threshold, color="red", linestyle="--", lw=2, label=f"Threshold={threshold:.4f}")
    for inj in INJECT_AT:
        ax.axvline(inj, color="orange", alpha=0.5, lw=2)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("MSE")
    ax.set_title("Sensor 2 — Reconstruction Error Timeline (anomalies injected at orange lines)",
                 fontweight="bold")
    ax.legend()
    plt.tight_layout()
    plt.savefig("../logs/kafka_error_timeline.png", dpi=120, bbox_inches="tight")
    plt.show()

## 5. FastAPI Endpoint Test

Tests the REST API using the `httpx` test client (no live server required).

In [ ]:
if not os.path.exists(MODEL_PATH):
    print("⚠️  Model not found. Run notebook 03_training.ipynb first.")
else:
    from fastapi.testclient import TestClient
    from src.api.main import app
    from src.api.inference import InferenceEngine

    # Load model into the singleton
    InferenceEngine.load(
        model_path=MODEL_PATH,
        scaler_path=SCALER_PATH,
        threshold_path=THRESHOLD_PATH,
    )

    client = TestClient(app)

    # ── Health check ──────────────────────────────────────────────────
    resp = client.get("/health")
    assert resp.status_code == 200
    health = resp.json()
    print(f"✅ /health  → {health['status']}  threshold={health['threshold']:.6f}")

    # ── Normal sequence prediction ────────────────────────────────────
    normal_seq = gen.generate_normal_sequence().tolist()
    payload = {"sensor_id": 10, "sequence": normal_seq}
    resp = client.post("/predict", json=payload)
    assert resp.status_code == 200
    result = resp.json()
    print(f"\n✅ /predict (normal):")
    print(f"   is_anomaly : {result['is_anomaly']}")
    print(f"   error      : {result['reconstruction_error']:.6f}")
    print(f"   severity   : {result['severity']}")
    print(f"   latency_ms : {result.get('inference_ms', 'N/A')}")

    # ── Anomaly sequence prediction ───────────────────────────────────
    anomaly_seq = gen.inject_anomaly(
        gen.generate_normal_sequence(), anomaly_type="spike"
    ).tolist()
    payload_anom = {"sensor_id": 15, "sequence": anomaly_seq}
    resp_anom = client.post("/predict", json=payload_anom)
    assert resp_anom.status_code == 200
    result_anom = resp_anom.json()
    print(f"\n✅ /predict (anomaly spike):")
    print(f"   is_anomaly : {result_anom['is_anomaly']}")
    print(f"   error      : {result_anom['reconstruction_error']:.6f}")
    print(f"   severity   : {result_anom['severity']}")

    # ── Alerts endpoint ───────────────────────────────────────────────
    resp_alerts = client.get("/alerts/recent")
    assert resp_alerts.status_code == 200
    alerts_data = resp_alerts.json()
    print(f"\n✅ /alerts/recent → total={alerts_data['total']}")

    # ── Metrics endpoint ──────────────────────────────────────────────
    resp_metrics = client.get("/metrics")
    assert resp_metrics.status_code == 200
    metrics = resp_metrics.json()
    print(f"\n✅ /metrics:")
    for k, v in metrics.items():
        print(f"   {k}: {v}")

    print("\n→ NEXT STEP: Run notebook 05_monitoring.ipynb")